# Prediction du Diabete — Pipeline MLOps avec SeraPlot

Analyse complete du dataset PIMA Indians Diabetes.  
Exploration des donnees, entrainement RandomForest + GridSearchCV, evaluation et prediction manuelle.  
Toutes les visualisations sont realisees exclusivement avec **SeraPlot**.

---

| | |
|---|---|
| **Dataset** | 768 patientes, 8 variables medicales |
| **Cible** | Outcome — 0 = Non-Diabetique, 1 = Diabetique |
| **Modele** | RandomForestClassifier (SeraPlot) |
| **Optimisation** | GridSearchCV 192 combinaisons, 5-fold CV |

In [5]:
%pip install --force-reinstall --no-deps seraplot==2.3.83

   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.3 MB 12.5 MB/s eta 0:00:01
   ----------- ---------------------------- 3.4/12.3 MB 10.1 MB/s eta 0:00:01
   ------------- -------------------------- 4.2/12.3 MB 10.9 MB/s eta 0:00:01
   ----------------- ---------------------- 5.2/12.3 MB 7.2 MB/s eta 0:00:01
   ----------------------- ---------------- 7.3/12.3 MB 7.5 MB/s eta 0:00:01
   ------------------------------ --------- 9.4/12.3 MB 8.3 MB/s eta 0:00:01
   -------------------------------------- - 11.8/12.3 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 8.6 MB/s  0:00:01
  Attempting uninstall: seraplot
    Found existing installation: seraplot 2.3.82
    Uninstalling seraplot-2.3.82:
      Successfully uninstalled seraplot-2.3.82
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

{'status': 'ok', 'restart': True}

: 

In [1]:
import sys
import os

sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import seraplot as sp
from IPython.display import HTML, display

from src.config import FEATURE_COLS, TARGET_COL, RANDOM_SEED, CV_FOLDS, MODEL_NAME
from src.data_prep import DataPipeline
from src.evaluate import ClassificationEvaluator
from src.model_store import ModelRegistry

print('SeraPlot version :', sp.__version__)

SeraPlot version : 2.3.83


In [2]:
dp = DataPipeline()
df = dp.clean().dataframe.copy()

print(f'Dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes')
df.head()

Dataset : 768 lignes x 9 colonnes


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0.0,33.6,0.627,50,1
1,1,85,66,29,0.0,26.6,0.351,31,0
2,8,183,64,0,0.0,23.3,0.672,32,1
3,1,89,66,23,94.0,28.1,0.167,21,0
4,0,137,40,35,168.0,43.1,1.200,33,1


## 1. Exploration des donnees

In [3]:
counts = df[TARGET_COL].value_counts().sort_index()
labels_cls = ['Non-Diabetique', 'Diabetique']
values_cls = [float(counts[0]), float(counts[1])]

print(f'Non-Diabetique : {int(counts[0])} cas  ({counts[0]/len(df):.1%})')
print(f'Diabetique     : {int(counts[1])} cas  ({counts[1]/len(df):.1%})')

chart = sp.build_donut_chart(
    'Repartition des classes',
    labels_cls, values_cls,
    width=700, height=420
)

Non-Diabetique : 500 cas  (65.1%)
Diabetique     : 268 cas  (34.9%)


c:\Users\Quentin\Desktop\SeraPlot\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [4]:
chart = sp.build_bar_chart(
    'Nombre de cas par classe',
    labels_cls, values_cls,
    show_text=True,
    y_label='Nombre de patients',
    width=620, height=380
)

### Distributions des variables par classe

Histogrammes superposes pour chaque variable medicale — bleu = Non-Diabetique, rose = Diabetique.

In [5]:
feat_list = list(FEATURE_COLS)
non_diab = df[df[TARGET_COL] == 0]
diab = df[df[TARGET_COL] == 1]

for feat in feat_list:
    nd_vals = non_diab[feat].tolist()
    d_vals = diab[feat].tolist()
    chart = sp.build_histogram_overlay(
        f'Distribution : {feat}',
        values=nd_vals, overlay=d_vals,
        series_names=['Non-Diabetique', 'Diabetique'],
        x_label=feat,
        y_label='Nombre de patients',
        width=860, height=360
    )

### Boxplots — dispersion des valeurs par classe

In [6]:
cats_cls = df[TARGET_COL].map({0: 'Non-Diabetique', 1: 'Diabetique'}).tolist()

for feat in feat_list:
    vals = df[feat].tolist()
    chart = sp.build_boxplot(
        f'Boxplot : {feat}',
        categories=cats_cls, values=vals,
        x_label='Classe', y_label=feat,
        width=700, height=420
    )
    display(HTML(chart.html))

### Matrice de correlation

In [7]:
corr = df[feat_list].corr()
flat_corr = corr.values.flatten().tolist()

chart = sp.build_heatmap(
    'Matrice de correlation des variables',
    feat_list, flat_corr,
    col_labels=feat_list,
    show_values=True,
    width=820, height=560
)
display(HTML(chart.html))

c:\Users\Quentin\Desktop\SeraPlot\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


### Densites et distributions avancees

In [8]:
glucose_vals = df['Glucose'].tolist()

chart = sp.build_kde_chart(
    'Densite du Glucose par classe',
    values=glucose_vals, categories=cats_cls,
    x_label='Glucose', filled=True,
    width=860, height=380
)
display(HTML(chart.html))

bmi_vals = df['BMI'].tolist()

chart = sp.build_scatter_chart(
    'Glucose vs BMI — colore par classe',
    x=glucose_vals, y=bmi_vals,
    color_groups=cats_cls,
    x_label='Glucose', y_label='BMI',
    gridlines=True,
    width=900, height=540
)
display(HTML(chart.html))

In [9]:
chart = sp.build_violin(
    'Distribution du BMI par classe',
    categories=cats_cls, values=bmi_vals,
    x_label='Classe', y_label='BMI',
    width=700, height=460
)
display(HTML(chart.html))

age_vals = df['Age'].tolist()

chart = sp.build_ridgeline_chart(
    'Distribution de Age par classe',
    values=age_vals, categories=cats_cls,
    x_label='Age',
    overlap=0.6,
    width=820, height=500
)
display(HTML(chart.html))

In [10]:
age_vals_3d = df['Age'].tolist()

chart = sp.build_scatter3d_chart(
    'Scatter 3D : Glucose x BMI x Age',
    x=glucose_vals, y=bmi_vals, z=age_vals_3d,
    color_labels=cats_cls,
    x_label='Glucose', y_label='BMI', z_label='Age',
    width=900, height=580
)
display(HTML(chart.html))

## 2. Entrainement du modele

GridSearchCV sur 192 combinaisons de parametres (n_estimators x max_depth x min_samples_split x min_samples_leaf),  
avec validation croisee 5-fold. Le pipeline nettoie, divise et normalise les donnees automatiquement.

In [11]:
from src.train import TrainingPipeline

result = TrainingPipeline().run()

version = result['model_version']
cv_score = result['best_cv_score']
best_params = result['best_params']

print(f'Version      : {version}')
print(f'Meilleur CV  : {cv_score:.4f}')
print(f'Parametres   : {best_params}')
print()
print(f'Train  acc   : {result["train_metrics"]["accuracy"]:.4f}')
print(f'Val    acc   : {result["val_metrics"]["accuracy"]:.4f}')
print(f'Test   acc   : {result["test_metrics"]["accuracy"]:.4f}')
print(f'Test   F1    : {result["test_metrics"]["f1"]:.4f}')
print(f'Test   prec  : {result["test_metrics"]["precision"]:.4f}')
print(f'Test   rappel: {result["test_metrics"]["recall"]:.4f}')

c:\Users\Quentin\Desktop\SeraPlot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Quentin\Desktop\SeraPlot\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Version      : 1776940497
Meilleur CV  : 0.7900
Parametres   : {'min_samples_split': 2, 'n_estimators': 20, 'min_samples_leaf': 2, 'max_depth': 5}

Train  acc   : 0.8643
Val    acc   : 0.6957
Test   acc   : 0.6957
Test   F1    : 0.4615
Test   prec  : 0.5000
Test   rappel: 0.4286


In [12]:
metric_keys = ['accuracy', 'f1', 'precision', 'recall']
metric_labels = ['Accuracy', 'F1 Score', 'Precision', 'Rappel']

train_vals_m = [result['train_metrics'][k] for k in metric_keys]
val_vals_m   = [result['val_metrics'][k] for k in metric_keys]
test_vals_m  = [result['test_metrics'][k] for k in metric_keys]

series_values_m = train_vals_m + val_vals_m + test_vals_m

chart = sp.build_grouped_bar(
    'Metriques de performance : Train / Validation / Test',
    metric_labels,
    series_values_m,
    series_names=['Train', 'Validation', 'Test'],
    show_values=True,
    y_label='Score',
    gridlines=True,
    width=1050, height=480
)
display(HTML(chart.html))

c:\Users\Quentin\Desktop\SeraPlot\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [13]:
dp2 = DataPipeline()
ds2 = dp2.clean().split()
scaled2 = DataPipeline.scale(ds2.x_train, ds2.x_val, ds2.x_test)

cv_result = ClassificationEvaluator.cross_validate(
    MODEL_NAME, scaled2.x_train, ds2.y_train, cv=CV_FOLDS, seed=RANDOM_SEED
)

cv_scores = cv_result['cv_scores']
fold_labels = [f'Fold {i+1}' for i in range(len(cv_scores))]
cv_mean = cv_result['cv_mean']
cv_std = cv_result['cv_std']

chart = sp.build_line_chart(
    f'Scores de validation croisee  (mean={cv_mean:.3f}  std={cv_std:.3f})',
    fold_labels, [float(s) for s in cv_scores],
    show_points=True,
    x_label='Fold', y_label='Accuracy',
    gridlines=True,
    width=800, height=400
)
display(HTML(chart.html))

c:\Users\Quentin\Desktop\SeraPlot\.venv\Lib\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## 3. Evaluation du modele

In [14]:
cm = result['test_metrics']['confusion_matrix']
flat_cm = [float(cm[0][0]), float(cm[0][1]), float(cm[1][0]), float(cm[1][1])]

row_labels_cm = ['Reel : Non-Diabetique', 'Reel : Diabetique']
col_labels_cm = ['Predit : Non-Diabetique', 'Predit : Diabetique']

chart = sp.build_heatmap(
    'Matrice de Confusion (Test set)',
    row_labels_cm, flat_cm,
    col_labels=col_labels_cm,
    show_values=True,
    color_low=0x6366F1,
    color_mid=0xfafbfc,
    color_high=0xF43F5E,
    width=740, height=440
)
display(HTML(chart.html))

print(result['test_metrics']['classification_report'])

              precision    recall  f1-score   support

           0    0.7647    0.8125    0.7879       80
           1    0.5000    0.4286    0.4615       35

    accuracy                        0.6957      115
   macro avg    0.6324    0.6205    0.6247      115
weighted avg    0.6841    0.6957    0.6886      115



In [15]:
model_fi = sp.RandomForestClassifier(**{k: int(v) for k, v in best_params.items()})
model_fi.fit(scaled2.x_train, ds2.y_train)

importances = model_fi.feature_importances_

chart = sp.build_hbar(
    'Importance des variables',
    feat_list,
    [float(v) for v in importances],
    show_text=True,
    sort_order='desc',
    x_label="Score d'importance",
    width=820, height=480
)
display(HTML(chart.html))

chart2 = sp.build_lollipop_chart(
    'Importance des variables (lollipop)',
    feat_list,
    [float(v) for v in importances],
    sort_order='desc',
    x_label='Variable', y_label='Score',
    width=820, height=440
)
display(HTML(chart2.html))

In [16]:
metric_axes = ['Accuracy', 'F1 Score', 'Precision', 'Rappel']
metric_keys_r = ['accuracy', 'f1', 'precision', 'recall']

train_r = [result['train_metrics'][k] for k in metric_keys_r]
val_r   = [result['val_metrics'][k] for k in metric_keys_r]
test_r  = [result['test_metrics'][k] for k in metric_keys_r]

chart = sp.build_radar_chart(
    'Radar des metriques : Train / Validation / Test',
    metric_axes,
    [train_r, val_r, test_r],
    series_names=['Train', 'Validation', 'Test'],
    filled=True,
    fill_opacity=40,
    width=700, height=560
)
display(HTML(chart.html))

In [17]:
phases = ['Train', 'Validation', 'Test']
correct_counts = [
    round(result['train_metrics']['accuracy'] * len(ds2.y_train)),
    round(result['val_metrics']['accuracy'] * len(ds2.y_val)),
    round(result['test_metrics']['accuracy'] * len(ds2.y_test))
]
total_counts = [len(ds2.y_train), len(ds2.y_val), len(ds2.y_test)]
wrong_counts = [t - c for t, c in zip(total_counts, correct_counts)]

series_vals = [float(v) for v in correct_counts + wrong_counts]

chart = sp.build_stacked_bar(
    'Predictions correctes vs incorrectes par phase',
    phases,
    series_vals,
    series_names=['Correctes', 'Incorrectes'],
    show_values=True,
    y_label='Nombre de patients',
    width=800, height=440
)
display(HTML(chart.html))

## 4. Prediction manuelle

Modifiez les valeurs du patient ci-dessous pour predire si oui ou non il est diabetique.

In [18]:
registry = ModelRegistry()
config_store = registry.load('latest')

scaler_mean  = np.array(config_store['scaler']['mean'], dtype=np.float64)
scaler_scale = np.array(config_store['scaler']['scale'], dtype=np.float64)

best_p = config_store['params']
model_pred = sp.RandomForestClassifier(**{k: int(v) for k, v in best_p.items()})
model_pred.fit(scaled2.x_train, ds2.y_train)

patient = {
    'Pregnancies':              6,
    'Glucose':                  148,
    'BloodPressure':            72,
    'SkinThickness':            35,
    'Insulin':                  0,
    'BMI':                      33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age':                      50,
}

x_raw = np.array([[patient[f] for f in FEATURE_COLS]], dtype=np.float64)
x_sc  = (x_raw - scaler_mean) / scaler_scale

pred  = model_pred.predict(x_sc)
proba = model_pred.predict_proba(x_sc)

proba_nd = float(proba[0][0])
proba_d  = float(proba[0][1])
label    = 'Diabetique' if pred[0] == 1 else 'Non-Diabetique'

print('=== RESULTAT DE PREDICTION ===')
print(f'Patient         : {patient}')
print()
print(f'Prediction      : {label}')
print(f'P(Diabetique)   : {proba_d:.1%}')
print(f'P(Non-Diab.)    : {proba_nd:.1%}')

=== RESULTAT DE PREDICTION ===
Patient         : {'Pregnancies': 6, 'Glucose': 148, 'BloodPressure': 72, 'SkinThickness': 35, 'Insulin': 0, 'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 50}

Prediction      : Diabetique
P(Diabetique)   : 77.9%
P(Non-Diab.)    : 22.1%


In [19]:
chart = sp.build_pie_chart(
    f'Probabilites de prediction — {label}',
    ['Non-Diabetique', 'Diabetique'],
    [proba_nd, proba_d],
    width=640, height=400
)
display(HTML(chart.html))

chart2 = sp.build_gauge(
    'Risque de diabete',
    proba_d * 100,
    min_val=0.0, max_val=100.0,
    label=f'{proba_d:.1%}',
    width=440, height=320
)
display(HTML(chart2.html))

### Predictions sur plusieurs profils types

In [20]:
profils = [
    {'nom': 'Profil sain',       'Pregnancies': 1,  'Glucose': 89,  'BloodPressure': 66, 'SkinThickness': 23, 'Insulin': 94,  'BMI': 28.1, 'DiabetesPedigreeFunction': 0.167, 'Age': 21},
    {'nom': 'Profil moyen',      'Pregnancies': 3,  'Glucose': 120, 'BloodPressure': 70, 'SkinThickness': 25, 'Insulin': 80,  'BMI': 30.5, 'DiabetesPedigreeFunction': 0.350, 'Age': 35},
    {'nom': 'Profil a risque',   'Pregnancies': 6,  'Glucose': 148, 'BloodPressure': 72, 'SkinThickness': 35, 'Insulin': 0,   'BMI': 33.6, 'DiabetesPedigreeFunction': 0.627, 'Age': 50},
    {'nom': 'Profil severe',     'Pregnancies': 10, 'Glucose': 168, 'BloodPressure': 74, 'SkinThickness': 40, 'Insulin': 350, 'BMI': 38.0, 'DiabetesPedigreeFunction': 0.900, 'Age': 58},
]

profil_labels = []
profil_probas = []

for profil in profils:
    nom = profil['nom']
    x_p = np.array([[profil[f] for f in FEATURE_COLS]], dtype=np.float64)
    x_p_sc = (x_p - scaler_mean) / scaler_scale
    pred_p = model_pred.predict(x_p_sc)
    prob_p = model_pred.predict_proba(x_p_sc)
    p_d = float(prob_p[0][1])
    label_p = 'Diabetique' if pred_p[0] == 1 else 'Non-Diabetique'
    profil_labels.append(f'{nom}')
    profil_probas.append(p_d * 100)
    print(f'{nom:25s}  =>  {label_p}  (P_diabete={p_d:.1%})')

chart = sp.build_bar_chart(
    'Probabilite de diabete par profil (%)',
    profil_labels, profil_probas,
    show_text=True,
    y_label='Probabilite (%)',
    sort_order='desc',
    width=800, height=420
)
display(HTML(chart.html))

Profil sain                =>  Non-Diabetique  (P_diabete=2.4%)
Profil moyen               =>  Diabetique  (P_diabete=51.8%)
Profil a risque            =>  Diabetique  (P_diabete=77.9%)
Profil severe              =>  Diabetique  (P_diabete=86.1%)
